# v24 — Type-aware verifier/reranker, no training

Notebook này **không train** và **không generate lại** mặc định. Nó đọc `v23_val_candidates.json` đã upload vào dataset, rồi thử các chiến lược rerank theo question type:

- MC: ưu tiên candidate A/B/C/D, phạt No/Unknown.
- YNU: ưu tiên Yes/No/Unknown, chống spam No/Unknown.
- Đo first / majority / type-aware rerank / grid-best / oracle.
- Nếu MC oracle thấp, notebook có scaffold cho bước tiếp theo: option-wise MC entailment.

In [ ]:
# ==================================================================
# STAGE 0 -- Config
# ==================================================================
import os, re, json, math, random, statistics, traceback
from pathlib import Path
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)

BASE_DATA_DIR = Path("/kaggle/input/datasets/nguyenminhtric/test-pipeline")
V23_DIR = Path("/kaggle/input/datasets/yixuanisthebest/v2333333")

CAND_PATH = V23_DIR / "v23_val_candidates.json"
OLD_SUMMARY_PATH = V23_DIR / "v23_val_summary.json"
ADAPTER_DIR = V23_DIR

OUT_DIR = Path("/kaggle/working/v24_typeaware_rerank")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_OPTIONWISE_MC = False
OPTIONWISE_MAX_MC = None
OPTIONWISE_BON_N = 1
OPTIONWISE_MAX_NEW_TOKENS = 160

LABELS = ["A", "B", "C", "D", "Yes", "No", "Unknown"]
MC_LABELS = ["A", "B", "C", "D"]
YNU_LABELS = ["Yes", "No", "Unknown"]

F1_GATES = {"A": 0.55, "B": 0.50, "C": 0.40, "D": 0.35, "Yes": 0.60, "No": 0.60, "Unknown": 0.55}
RECALL_GATE = 0.30
MACRO_F1_GATE = 0.60
MC_MACRO_F1_GATE = 0.65

print("Config OK")
print("CAND_PATH:", CAND_PATH, "exists=", CAND_PATH.exists())
print("ADAPTER_DIR:", ADAPTER_DIR, "exists=", ADAPTER_DIR.exists())
print("OUT_DIR:", OUT_DIR)

In [ ]:
# ==================================================================
# STAGE 1 -- Load saved VAL candidates
# ==================================================================
assert CAND_PATH.exists(), f"Missing candidate file: {CAND_PATH}"

with open(CAND_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print("Loaded results:", len(results))
print("First result keys:", sorted(results[0].keys()))
if results and results[0].get("candidates"):
    print("First candidate keys:", sorted(results[0]["candidates"][0].keys()))

if OLD_SUMMARY_PATH.exists():
    try:
        with open(OLD_SUMMARY_PATH, "r", encoding="utf-8") as f:
            old_summary = json.load(f)
        print("Loaded old summary keys:", list(old_summary.keys())[:20])
    except Exception as e:
        old_summary = None
        print("Could not load old summary:", repr(e))
else:
    old_summary = None

for i, r in enumerate(results[:2]):
    print("\n--- sample", i, "---")
    print("gold:", r.get("gold"), "q_type:", r.get("q_type"), "question:", str(r.get("question", ""))[:140])
    for c in r.get("candidates", [])[:3]:
        print(" cand", c.get("candidate_idx"), "ans=", c.get("answer"), "supp=", c.get("supporting_premises"))

In [ ]:
# ==================================================================
# STAGE 2 -- Utilities
# ==================================================================
def norm_ans(x):
    if x is None:
        return "UNPARSEABLE"
    s = str(x).strip()
    u = s.upper().strip(" .,:;")
    if u in {"A", "B", "C", "D"}: return u
    if u in {"YES", "Y", "TRUE"}: return "Yes"
    if u in {"NO", "N", "FALSE"}: return "No"
    if u in {"UNKNOWN", "UNCERTAIN", "UNSURE", "CANNOT BE DETERMINED", "UNDETERMINED"}: return "Unknown"
    return "UNPARSEABLE"

def infer_qtype(r):
    qt = str(r.get("q_type", "")).lower()
    if "mc" in qt or "multiple" in qt: return "mc"
    if "yes" in qt or "yn" in qt or "unknown" in qt: return "ynu"
    gold = norm_ans(r.get("gold"))
    if gold in MC_LABELS: return "mc"
    if gold in YNU_LABELS: return "ynu"
    q = str(r.get("question", ""))
    if re.search(r"\bA[.)]\s+.+\bB[.)]\s+", q, flags=re.S): return "mc"
    return "ynu"

def raw_text(c):
    for k in ["raw", "text", "output", "generation", "explanation"]:
        if k in c and c[k]: return str(c[k])
    return ""

def get_supp(c):
    sp = c.get("supporting_premises", c.get("supporting", c.get("supp", [])))
    if sp is None: return []
    if isinstance(sp, str): return [int(x) for x in re.findall(r"\d+", sp)]
    if isinstance(sp, (list, tuple)):
        out = []
        for x in sp:
            try: out.append(int(x))
            except Exception: pass
        return out
    return []

def premise_count(r):
    for k in ["premises", "premises-NL", "premises_NL", "premises_nl"]:
        v = r.get(k)
        if isinstance(v, list): return len(v)
    mx = 0
    for c in r.get("candidates", []):
        for s in get_supp(c): mx = max(mx, s)
    return mx if mx > 0 else 999

def citation_stats_for_candidate(c, r):
    supp = get_supp(c); n_prem = premise_count(r)
    if not supp:
        return {"has_citation": False, "valid_citation": False, "n_cited": 0, "too_many": False, "invalid_count": 0}
    invalid = [x for x in supp if x < 1 or x > n_prem]
    return {"has_citation": True, "valid_citation": len(invalid)==0, "n_cited": len(set(supp)), "too_many": len(set(supp))>8, "invalid_count": len(invalid)}

def answer_counts(cands):
    return Counter(norm_ans(c.get("answer")) for c in cands)

def candidate_base_features(c, r, cnt):
    a = norm_ans(c.get("answer")); n = max(1, sum(cnt.values()))
    cite = citation_stats_for_candidate(c, r); txt = raw_text(c)
    final_mentions = len(re.findall(r"Final\s+Answer\s*:", txt, flags=re.I))
    clean_final = (final_mentions == 1) or (a != "UNPARSEABLE" and final_mentions <= 1)
    return {"answer": a, "vote_share": cnt[a]/n, "vote_count": cnt[a], "has_citation": cite["has_citation"],
            "valid_citation": cite["valid_citation"], "n_cited": cite["n_cited"], "too_many_citations": cite["too_many"],
            "invalid_citation_count": cite["invalid_count"], "clean_final": clean_final, "text_len": len(txt.split())}

def choose_first(r):
    cands = r.get("candidates", [])
    return norm_ans(cands[0].get("answer")) if cands else "UNPARSEABLE"

def choose_majority(r, prefer_type=True):
    cands = r.get("candidates", [])
    if not cands: return "UNPARSEABLE"
    qt = infer_qtype(r); cnt = answer_counts(cands); items = list(cnt.items())
    def type_bonus(a):
        if qt == "mc" and a in MC_LABELS: return 1
        if qt == "ynu" and a in YNU_LABELS: return 1
        return 0
    items.sort(key=lambda kv: (kv[1], type_bonus(kv[0])), reverse=True)
    return items[0][0]

def choose_oracle(r):
    gold = norm_ans(r.get("gold"))
    for c in r.get("candidates", []):
        if norm_ans(c.get("answer")) == gold: return gold
    return choose_first(r)

def f1_pr_recall(y_true, y_pred, labels=LABELS):
    rows = {}
    for lab in labels:
        tp = sum(1 for g,p in zip(y_true,y_pred) if g==lab and p==lab)
        fp = sum(1 for g,p in zip(y_true,y_pred) if g!=lab and p==lab)
        fn = sum(1 for g,p in zip(y_true,y_pred) if g==lab and p!=lab)
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        rows[lab] = {"precision": prec, "recall": rec, "f1": f1, "support": sum(1 for g in y_true if g==lab), "pred_count": sum(1 for p in y_pred if p==lab), "tp": tp, "fp": fp, "fn": fn}
    return rows

def metric_summary(y_true, y_pred, title="METRICS", print_rows=True):
    n = len(y_true); acc = sum(g==p for g,p in zip(y_true,y_pred))/n if n else 0.0
    rows = f1_pr_recall(y_true,y_pred)
    macro = sum(rows[l]["f1"] for l in LABELS)/len(LABELS)
    weighted = sum(rows[l]["f1"]*rows[l]["support"] for l in LABELS)/max(1,n)
    mc_idx = [i for i,g in enumerate(y_true) if g in MC_LABELS]; ynu_idx = [i for i,g in enumerate(y_true) if g in YNU_LABELS]
    sub_acc = lambda idx: sum(y_true[i]==y_pred[i] for i in idx)/len(idx) if idx else 0.0
    mc_macro = sum(rows[l]["f1"] for l in MC_LABELS)/4; ynu_macro = sum(rows[l]["f1"] for l in YNU_LABELS)/3
    out = {"n": n, "acc": acc, "macro_f1": macro, "weighted_f1": weighted, "mc_acc": sub_acc(mc_idx), "mc_macro_f1": mc_macro, "mc_n": len(mc_idx), "ynu_acc": sub_acc(ynu_idx), "ynu_macro_f1": ynu_macro, "ynu_n": len(ynu_idx), "labels": rows, "gold_dist": dict(Counter(y_true)), "pred_dist": dict(Counter(y_pred))}
    print("="*100); print(title); print("="*100)
    print(f"N={n} acc={acc:.3f} macro_f1={macro:.3f} weighted_f1={weighted:.3f}")
    print(f"MC:  acc={out['mc_acc']:.3f}  macro_f1={mc_macro:.3f}  n={len(mc_idx)}")
    print(f"YNU: acc={out['ynu_acc']:.3f}  macro_f1={ynu_macro:.3f}  n={len(ynu_idx)}")
    if print_rows:
        print("-"*100); print(f"{'Label':10s} {'P':>8s} {'R':>8s} {'F1':>8s} {'Gold':>8s} {'Pred':>8s}")
        for lab in LABELS:
            rr=rows[lab]; print(f"{lab:10s} {rr['precision']:8.3f} {rr['recall']:8.3f} {rr['f1']:8.3f} {rr['support']:8d} {rr['pred_count']:8d}")
        print("Gold dist:", out["gold_dist"]); print("Pred dist:", out["pred_dist"])
    return out

def gate_report(metrics):
    fails=[]
    for lab,thr in F1_GATES.items():
        val=metrics["labels"][lab]["f1"]
        if val<thr: fails.append((lab,"f1",val,thr))
        rec=metrics["labels"][lab]["recall"]
        if rec<RECALL_GATE: fails.append((lab,"recall",rec,RECALL_GATE))
    if metrics["mc_macro_f1"]<MC_MACRO_F1_GATE: fails.append(("MC","macro_f1",metrics["mc_macro_f1"],MC_MACRO_F1_GATE))
    if metrics["macro_f1"]<MACRO_F1_GATE: fails.append(("ALL","macro_f1",metrics["macro_f1"],MACRO_F1_GATE))
    print("GATE:", "PASS" if not fails else "FAIL")
    for x in fails: print(" ", x)
    return fails
print("Utilities OK")

In [ ]:
# ==================================================================
# STAGE 3 -- Rerank strategies
# ==================================================================
def score_candidate_typeaware(c, r, params):
    qt=infer_qtype(r); cands=r.get("candidates", []); cnt=answer_counts(cands); f=candidate_base_features(c,r,cnt); a=f["answer"]
    score=0.0
    score += params.get("vote_w",1.0)*f["vote_share"]
    score += params.get("citation_w",0.5)*float(f["valid_citation"])
    score += params.get("clean_final_w",0.2)*float(f["clean_final"])
    if 1 <= f["n_cited"] <= 5: score += params.get("short_support_bonus",0.2)
    if f["too_many_citations"]: score -= params.get("too_many_penalty",0.3)
    if f["invalid_citation_count"]>0: score -= params.get("invalid_citation_penalty",1.0)
    if qt=="mc":
        if a in MC_LABELS: score += params.get("mc_answer_bonus",1.0)
        elif a in YNU_LABELS: score -= params.get("mc_ynu_penalty",1.0)
        else: score -= 2.0
        if a in {"C","D"}: score += params.get("rare_mc_bonus",0.0)
    else:
        if a in YNU_LABELS: score += params.get("ynu_answer_bonus",0.5)
        elif a in MC_LABELS: score -= params.get("ynu_mc_penalty",1.0)
        else: score -= 2.0
        if a=="Yes": score += params.get("yes_bonus",0.0)
        if a=="No": score -= params.get("no_penalty",0.0)
        if a=="Unknown": score -= params.get("unknown_penalty",0.0)
    if a=="UNPARSEABLE": score -= 3.0
    return score

DEFAULT_PARAMS={"vote_w":1.0,"citation_w":0.4,"clean_final_w":0.2,"short_support_bonus":0.2,"too_many_penalty":0.3,"invalid_citation_penalty":1.0,"mc_answer_bonus":1.4,"mc_ynu_penalty":1.4,"rare_mc_bonus":0.1,"ynu_answer_bonus":0.7,"ynu_mc_penalty":1.2,"yes_bonus":0.6,"no_penalty":0.6,"unknown_penalty":0.4}

def choose_rerank(r, params=DEFAULT_PARAMS, return_candidate=False):
    cands=r.get("candidates", [])
    if not cands: return ("UNPARSEABLE", None) if return_candidate else "UNPARSEABLE"
    scored=[(score_candidate_typeaware(c,r,params), c) for c in cands]
    scored.sort(key=lambda x:x[0], reverse=True)
    best=scored[0][1]; ans=norm_ans(best.get("answer"))
    return (ans,best) if return_candidate else ans

def evaluate_method(name, chooser, subset=None):
    rows=results if subset is None else subset
    y_true=[norm_ans(r.get("gold")) for r in rows]
    y_pred=[chooser(r) for r in rows]
    m=metric_summary(y_true,y_pred,title=name)
    return m,y_pred
print("Rerank strategies OK")

In [ ]:
# ==================================================================
# STAGE 4 -- Baselines + default type-aware rerank
# ==================================================================
all_metrics={}; all_preds={}
methods=[("VAL -- FIRST CANDIDATE", choose_first), ("VAL -- MAJORITY VOTE", choose_majority), ("VAL -- TYPE-AWARE RERANK DEFAULT", lambda r: choose_rerank(r, DEFAULT_PARAMS)), ("VAL -- ORACLE AMONG CANDIDATES", choose_oracle)]
for name,chooser in methods:
    m,p=evaluate_method(name, chooser)
    all_metrics[name]=m; all_preds[name]=p
    gate_report(m); print()
print("\nKey diagnosis:")
for lab in LABELS:
    maj=all_metrics["VAL -- MAJORITY VOTE"]["labels"][lab]["recall"]
    rer=all_metrics["VAL -- TYPE-AWARE RERANK DEFAULT"]["labels"][lab]["recall"]
    ora=all_metrics["VAL -- ORACLE AMONG CANDIDATES"]["labels"][lab]["recall"]
    print(f"{lab:8s} majority_recall={maj:.3f}  rerank_recall={rer:.3f}  oracle_recall={ora:.3f}")

In [ ]:
# ==================================================================
# STAGE 5 -- Lightweight grid search on VAL to analyze useful signals
# NOTE: development only; do not claim as held-out tuning.
# ==================================================================
param_grid=[]
for yes_bonus in [0.2,0.5,0.8,1.2]:
    for no_penalty in [0.2,0.5,0.8,1.2]:
        for unk_penalty in [0.0,0.3,0.6,0.9]:
            for mc_bonus in [1.0,1.5,2.0]:
                p=dict(DEFAULT_PARAMS); p.update({"yes_bonus":yes_bonus,"no_penalty":no_penalty,"unknown_penalty":unk_penalty,"mc_answer_bonus":mc_bonus,"mc_ynu_penalty":mc_bonus})
                param_grid.append(p)

def metric_silent(y_true,y_pred):
    rows=f1_pr_recall(y_true,y_pred); n=len(y_true); acc=sum(g==p for g,p in zip(y_true,y_pred))/n if n else 0
    macro=sum(rows[l]["f1"] for l in LABELS)/len(LABELS); mc_macro=sum(rows[l]["f1"] for l in MC_LABELS)/4; ynu_macro=sum(rows[l]["f1"] for l in YNU_LABELS)/3
    objective=macro+0.25*mc_macro+0.15*ynu_macro
    return {"acc":acc,"macro":macro,"mc_macro":mc_macro,"ynu_macro":ynu_macro,"objective":objective,"rows":rows}

y_true=[norm_ans(r.get("gold")) for r in results]
best=None; grid_rows=[]
for p in param_grid:
    y_pred=[choose_rerank(r,p) for r in results]
    ms=metric_silent(y_true,y_pred)
    row={**{k:v for k,v in p.items() if k in ["yes_bonus","no_penalty","unknown_penalty","mc_answer_bonus"]}, **ms}
    grid_rows.append(row)
    if best is None or ms["objective"] > best["objective"]:
        best={**p, **ms, "pred": y_pred}
print("Grid tried:", len(param_grid))
print("Best objective:", best["objective"])
print("Best acc/macro/mc/ynu:", best["acc"], best["macro"], best["mc_macro"], best["ynu_macro"])
print("Best params subset:", {k:best[k] for k in ["yes_bonus","no_penalty","unknown_penalty","mc_answer_bonus","mc_ynu_penalty"]})
best_metrics=metric_summary(y_true,best["pred"],title="VAL -- GRID BEST TYPE-AWARE RERANK")
gate_report(best_metrics)
try:
    import pandas as pd
    df = pd.DataFrame([{k:v for k,v in row.items() if k!="rows"} for row in grid_rows]).sort_values("objective", ascending=False)
    df.to_csv(OUT_DIR/"v24_grid_search.csv", index=False)
    print("Saved grid:", OUT_DIR/"v24_grid_search.csv")
except Exception as e:
    print("Could not save grid csv:", repr(e))

In [ ]:
# ==================================================================
# STAGE 6 -- Failure analysis: correct candidate exists but rerank misses
# ==================================================================
def has_correct_candidate(r):
    gold=norm_ans(r.get("gold"))
    return any(norm_ans(c.get("answer"))==gold for c in r.get("candidates", []))
miss_cases=[]
for i,r in enumerate(results):
    gold=norm_ans(r.get("gold")); pred=best["pred"][i]
    if pred!=gold and has_correct_candidate(r): miss_cases.append((i,r,pred))
print("Cases where correct candidate exists but grid rerank misses:", len(miss_cases))
for k,(i,r,pred) in enumerate(miss_cases[:12],1):
    print("\n"+"="*120)
    print(f"[{k}] idx={i} q_type={infer_qtype(r)} gold={norm_ans(r.get('gold'))} pred={pred}")
    print("Question:", str(r.get("question", ""))[:500])
    cnt=answer_counts(r.get("candidates", [])); print("Candidate answer counts:", dict(cnt))
    scored=[]
    for c in r.get("candidates", []):
        ans=norm_ans(c.get("answer")); sc=score_candidate_typeaware(c,r,{kk:best[kk] for kk in DEFAULT_PARAMS.keys()})
        scored.append((sc,ans,get_supp(c),raw_text(c)[:220].replace("\n"," ")))
    for sc,ans,supp,txt in sorted(scored, reverse=True):
        mark="*" if ans==norm_ans(r.get("gold")) else " "
        print(f" {mark} score={sc:6.3f} ans={ans:10s} supp={supp} text={txt}")

In [ ]:
# ==================================================================
# STAGE 7 -- MC coverage diagnosis
# ==================================================================
mc_rows=[r for r in results if infer_qtype(r)=="mc"]
ynu_rows=[r for r in results if infer_qtype(r)=="ynu"]
print("MC rows:", len(mc_rows), "YNU rows:", len(ynu_rows))

def coverage_by_label(rows):
    out={}
    for lab in LABELS:
        sub=[r for r in rows if norm_ans(r.get("gold"))==lab]
        if not sub: continue
        cov=sum(has_correct_candidate(r) for r in sub)/len(sub)
        maj_rec=sum(choose_majority(r)==lab for r in sub)/len(sub)
        rer_rec=sum(choose_rerank(r,{kk:best[kk] for kk in DEFAULT_PARAMS.keys()})==lab for r in sub)/len(sub)
        out[lab]={"n":len(sub),"oracle_coverage":cov,"majority_recall":maj_rec,"rerank_recall":rer_rec}
    return out
cov=coverage_by_label(results)
for lab,row in cov.items():
    print(f"{lab:8s} n={row['n']:3d} oracle_cov={row['oracle_coverage']:.3f} majority_rec={row['majority_recall']:.3f} rerank_rec={row['rerank_recall']:.3f}")
print("\nInterpretation:")
print("- Low oracle coverage => generator rarely produces correct label; reranking cannot fix it.")
print("- High oracle coverage but low rerank recall => improve verifier/reranker.")

In [ ]:
# ==================================================================
# STAGE 8 -- Optional MC option-wise generation scaffold
# ==================================================================
def extract_options(question):
    q=str(question)
    pat=r"(?s)(?:^|\n|\s)([ABCD])[\.\)]\s*(.*?)(?=(?:\n|\s)[ABCD][\.\)]\s*|$)"
    matches=re.findall(pat,q)
    opts={}
    for lab,txt in matches:
        txt=txt.strip()
        if txt: opts[lab]=txt
    return opts

def build_optionwise_prompt(r,opt_label,opt_text):
    premises=r.get("premises") or r.get("premises-NL") or r.get("premises_NL") or []
    if isinstance(premises,list): prem_txt="\n".join([f"Premise {i+1}: {p}" for i,p in enumerate(premises)])
    else: prem_txt=str(premises)
    q=str(r.get("question", ""))
    return f"""You are a careful logic verifier.

Given the premises and one candidate option, decide whether the option is logically entailed by the premises.

Return exactly:
Reasoning: ...
Supporting Premises: [...]
Final Answer: Yes/No/Unknown

Premises:
{prem_txt}

Original question:
{q}

Candidate option {opt_label}: {opt_text}
"""

if RUN_OPTIONWISE_MC:
    print("RUN_OPTIONWISE_MC=True")
    print("This scaffold intentionally does not auto-run model loading.")
    print("Use v23 inference model-loading cell, then implement generate_text(prompt).")
    for r in mc_rows[:5]:
        print("\nQ:", str(r.get("question", ""))[:300])
        print("Options:", extract_options(r.get("question", "")))
else:
    print("RUN_OPTIONWISE_MC=False, skipping option-wise generation.")
    print("If MC oracle coverage is low, next notebook should implement this stage with model loading.")

In [ ]:
# ==================================================================
# STAGE 9 -- Save summary JSON/CSV
# ==================================================================
def to_jsonable(x):
    try:
        import numpy as np
        if isinstance(x,(np.integer,)): return int(x)
        if isinstance(x,(np.floating,)): return float(x)
        if isinstance(x,(np.bool_,)): return bool(x)
    except Exception: pass
    if isinstance(x,Path): return str(x)
    if isinstance(x,dict): return {str(k): to_jsonable(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)): return [to_jsonable(v) for v in x]
    return x
summary={"paths":{"cand_path":str(CAND_PATH),"adapter_dir":str(ADAPTER_DIR),"out_dir":str(OUT_DIR)},"metrics":{"first":all_metrics.get("VAL -- FIRST CANDIDATE"),"majority":all_metrics.get("VAL -- MAJORITY VOTE"),"default_typeaware":all_metrics.get("VAL -- TYPE-AWARE RERANK DEFAULT"),"grid_best":best_metrics,"oracle":all_metrics.get("VAL -- ORACLE AMONG CANDIDATES")},"best_params":{k:best[k] for k in DEFAULT_PARAMS.keys()},"coverage_by_label":cov,"notes":["No training or generation was run by default.","If MC oracle coverage is low, use option-wise MC entailment scoring next.","If oracle coverage is high but rerank low, improve verifier features."]}
summary_path=OUT_DIR/"v24_typeaware_rerank_summary.json"
with open(summary_path,"w",encoding="utf-8") as f: json.dump(to_jsonable(summary),f,ensure_ascii=False,indent=2)
print("Saved:", summary_path)
try:
    import pandas as pd
    table=[]
    for name,m in [("first",all_metrics.get("VAL -- FIRST CANDIDATE")),("majority",all_metrics.get("VAL -- MAJORITY VOTE")),("default_typeaware",all_metrics.get("VAL -- TYPE-AWARE RERANK DEFAULT")),("grid_best",best_metrics),("oracle",all_metrics.get("VAL -- ORACLE AMONG CANDIDATES"))]:
        if m:
            table.append({"method":name,"acc":m["acc"],"macro_f1":m["macro_f1"],"weighted_f1":m["weighted_f1"],"mc_macro_f1":m["mc_macro_f1"],"ynu_macro_f1":m["ynu_macro_f1"],"A_f1":m["labels"]["A"]["f1"],"B_f1":m["labels"]["B"]["f1"],"C_f1":m["labels"]["C"]["f1"],"D_f1":m["labels"]["D"]["f1"],"Yes_f1":m["labels"]["Yes"]["f1"],"No_f1":m["labels"]["No"]["f1"],"Unknown_f1":m["labels"]["Unknown"]["f1"]})
    df=pd.DataFrame(table)
    csv_path=OUT_DIR/"v24_metric_table.csv"
    df.to_csv(csv_path,index=False)
    print("Saved:", csv_path)
    display(df)
except Exception as e:
    print("CSV/display failed:", repr(e))